In [1]:
from pathlib import Path
import zipfile
import pandas as pd

# ---------------------------------------------------------------------
# Locate repo root robustly (works no matter where notebook is run from)
# ---------------------------------------------------------------------
def find_repo_root(start: Path | None = None) -> Path:
    p = start or Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        if p.parent == p:
            raise RuntimeError("Could not locate repo root (pyproject.toml not found)")
        p = p.parent
    return p

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

print("Repo root:", REPO_ROOT)
print("Raw data dir:", DATA_RAW)
print()

# ---------------------------------------------------------------------
# List available raw files
# ---------------------------------------------------------------------
raw_files = sorted(DATA_RAW.iterdir())

print("Raw files:")
for f in raw_files:
    print(f"  - {f.name}")


Repo root: /home/diggz/projects/carms-mini-platform
Raw data dir: /home/diggz/projects/carms-mini-platform/data/raw/dnokes

Raw files:
  - 1503_discipline.xlsx
  - 1503_markdown_program_descriptions.zip
  - 1503_markdown_program_descriptions_v2.zip
  - 1503_program_descriptions.zip
  - 1503_program_descriptions_v2.zip
  - 1503_program_descriptions_x_section.zip
  - 1503_program_master.xlsx
  - README.md
  - program_descriptions.zip


In [2]:
# ---------------------------------------------------------------------
# Load Excel files
# ---------------------------------------------------------------------
discipline_path = DATA_RAW / "1503_discipline.xlsx"
program_master_path = DATA_RAW / "1503_program_master.xlsx"

discipline_df = pd.read_excel(discipline_path)
program_df = pd.read_excel(program_master_path)

print("Discipline shape:", discipline_df.shape)
display(discipline_df.head())

print("\nProgram master shape:", program_df.shape)
display(program_df.head())


Discipline shape: (37, 2)


,discipline_id,discipline
0,13,Anesthesiology
1,96,Anesthesiology - Clinician Investigator Program
2,14,Cardiac Surgery
3,22,Dermatology
4,24,Diagnostic Radiology



Program master shape: (815, 11)


,Unnamed: 0,discipline_id,discipline_name,school_id,school_name,program_stream_id,program_stream_name,program_site,program_stream,program_name,program_url
0,0,13,Anesthesiology,5111821,Memorial University of Newfoundland,27447,St. John's - CMG Stream for CMG,St. John's,CMG Stream for CMG,Memorial University of Newfoundland / Anesthes...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
1,1,13,Anesthesiology,5177357,Dalhousie University,27416,Halifax - CMG Stream for CMG,Halifax,CMG Stream for CMG,Dalhousie University / Anesthesiology / Halifa...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
2,2,13,Anesthesiology,5963789,Université Laval,27032,Québec - Regular Stream for All,Québec,Regular Stream for All,Université Laval / Anesthesiology / Québec / R...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
3,3,13,Anesthesiology,6029325,Université de Sherbrooke,26782,Sherbrooke - Regular Stream for All,Sherbrooke,Regular Stream for All,Université de Sherbrooke / Anesthesiology / Sh...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
4,4,13,Anesthesiology,6094861,Université de Montréal,26747,Montreal - Regular Stream for All,Montreal,Regular Stream for All,Université de Montréal / Anesthesiology / Mont...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...


In [3]:
# ---------------------------------------------------------------------
# Inspect program description ZIPs
# ---------------------------------------------------------------------
zip_files = sorted(f for f in raw_files if f.suffix == ".zip")

for zpath in zip_files:
    print(f"\nZIP: {zpath.name}")
    with zipfile.ZipFile(zpath) as zf:
        names = zf.namelist()
        print(f"  Files: {len(names)}")
        for name in names[:5]:
            print("   ", name)
        if len(names) > 5:
            print("   ...")



ZIP: 1503_markdown_program_descriptions.zip
  Files: 1
    1503_markdown_program_descriptions.json

ZIP: 1503_markdown_program_descriptions_v2.zip
  Files: 1
    1503_markdown_program_descriptions_v2.json

ZIP: 1503_program_descriptions.zip
  Files: 1
    1503_program_descriptions.json

ZIP: 1503_program_descriptions_v2.zip
  Files: 1
    1503_program_descriptions_v2.json

ZIP: 1503_program_descriptions_x_section.zip
  Files: 1
    1503_program_descriptions_x_section.csv

ZIP: program_descriptions.zip
  Files: 815
    1503-27672.md
    1503-27674.md
    1503-27675.md
    1503-27676.md
    1503-27677.md
   ...


In [4]:
# ---------------------------------------------------------------------
# Peek a single description file
# ---------------------------------------------------------------------
sample_zip = zip_files[0]

with zipfile.ZipFile(sample_zip) as zf:
    first_file = zf.namelist()[0]
    print("Sample file:", first_file)
    content = zf.read(first_file).decode("utf-8", errors="replace")
    print("\n--- BEGIN SAMPLE ---\n")
    print(content[:1000])
    print("\n--- END SAMPLE ---")


Sample file: 1503_markdown_program_descriptions.json

--- BEGIN SAMPLE ---

[
    {
        "page_content": "#  Memorial University of Newfoundland - Anesthesiology - St. John's\n\n#  2025 R-1 Main Residency Match - first iteration  \nCMG Stream for CMG  \n\n##  Last approved on February 04, 2025\n\nSummary of changes\n\n## Approximate Quota:\n\n##   4\n\n##  Accreditation status : Accredited\n\n##  Provincial Criteria\n\nPrint copy link\n\nLink to display this program description  \n  \nTo copy link, press Ctrl-C on your keyboard or right-click on highlighted text\nand select Copy from the menu\n\n* * *\n\nProgram Director Dr. Sonia Sampson  \n\nAddress Discipline of Anesthesia, Room 1320  \nHealth Sciences Centre  \n300 Prince Philip Drive  \nSt. John's, NL , Newfoundland and Labrador, A1B 3V6  \n\nWork  (709) 864-3611\n\nWebsites of Interest Discipline of Anesthesia, Faculty of Medicine  \n\n* * *\n\n# Program Contacts\n\nName: Dr. Sonia Sampson\n\nTitle: Program Director\n\nEmail: 

In [6]:
# Cell 1: discover all raw files
from pathlib import Path
import pandas as pd
import zipfile

# Find repo root safely
def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    while not (p / "pyproject.toml").exists():
        if p.parent == p:
            raise RuntimeError("Could not find repo root")
        p = p.parent
    return p

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

print("Repo root:", REPO_ROOT)
print("Raw data dir:", DATA_RAW)
print("\nRaw files:")
for f in sorted(DATA_RAW.iterdir()):
    print(" -", f.name)


Repo root: /home/diggz/projects/carms-mini-platform
Raw data dir: /home/diggz/projects/carms-mini-platform/data/raw/dnokes

Raw files:
 - 1503_discipline.xlsx
 - 1503_markdown_program_descriptions.zip
 - 1503_markdown_program_descriptions_v2.zip
 - 1503_program_descriptions.zip
 - 1503_program_descriptions_v2.zip
 - 1503_program_descriptions_x_section.zip
 - 1503_program_master.xlsx
 - README.md
 - program_descriptions.zip


In [9]:
# Cell 2: Load and print disciplines
discipline_path = DATA_RAW / "1503_discipline.xlsx"

print("=== 1503_discipline.xlsx ===")
discipline_df = pd.read_excel(discipline_path)

print("Shape:", discipline_df.shape)
print("Columns:", list(discipline_df.columns))
display(discipline_df.head())

program_master_path = DATA_RAW / "1503_program_master.xlsx"

print("=== 1503_program_master.xlsx ===")
program_df = pd.read_excel(program_master_path)

print("Shape:", program_df.shape)
print("Columns:", list(program_df.columns))
display(program_df.head())

zip_files = sorted(DATA_RAW.glob("*.zip"))

print("ZIP files found:", len(zip_files))

for zpath in zip_files:
    print("\n=== ZIP:", zpath.name, "===")
    with zipfile.ZipFile(zpath) as zf:
        names = zf.namelist()
        print("File count:", len(names))
        for name in names[:10]:
            print(" ", name)
        if len(names) > 10:
            print("  ...")


for zpath in zip_files:
    print("\n=== SAMPLE FROM:", zpath.name, "===")
    with zipfile.ZipFile(zpath) as zf:
        names = zf.namelist()
        if not names:
            print("  (empty zip)")
            continue

        fname = names[0]
        print("Sample file:", fname)

        raw = zf.read(fname)
        try:
            text = raw.decode("utf-8")
            print(text[:1000])
        except UnicodeDecodeError:
            print("⚠️ Not UTF-8 text. Raw bytes length:", len(raw))


=== 1503_discipline.xlsx ===
Shape: (37, 2)
Columns: ['discipline_id', 'discipline']


,discipline_id,discipline
0,13,Anesthesiology
1,96,Anesthesiology - Clinician Investigator Program
2,14,Cardiac Surgery
3,22,Dermatology
4,24,Diagnostic Radiology


=== 1503_program_master.xlsx ===
Shape: (815, 11)
Columns: ['Unnamed: 0', 'discipline_id', 'discipline_name', 'school_id', 'school_name', 'program_stream_id', 'program_stream_name', 'program_site', 'program_stream', 'program_name', 'program_url']


,Unnamed: 0,discipline_id,discipline_name,school_id,school_name,program_stream_id,program_stream_name,program_site,program_stream,program_name,program_url
0,0,13,Anesthesiology,5111821,Memorial University of Newfoundland,27447,St. John's - CMG Stream for CMG,St. John's,CMG Stream for CMG,Memorial University of Newfoundland / Anesthes...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
1,1,13,Anesthesiology,5177357,Dalhousie University,27416,Halifax - CMG Stream for CMG,Halifax,CMG Stream for CMG,Dalhousie University / Anesthesiology / Halifa...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
2,2,13,Anesthesiology,5963789,Université Laval,27032,Québec - Regular Stream for All,Québec,Regular Stream for All,Université Laval / Anesthesiology / Québec / R...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
3,3,13,Anesthesiology,6029325,Université de Sherbrooke,26782,Sherbrooke - Regular Stream for All,Sherbrooke,Regular Stream for All,Université de Sherbrooke / Anesthesiology / Sh...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...
4,4,13,Anesthesiology,6094861,Université de Montréal,26747,Montreal - Regular Stream for All,Montreal,Regular Stream for All,Université de Montréal / Anesthesiology / Mont...,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...


ZIP files found: 6

=== ZIP: 1503_markdown_program_descriptions.zip ===
File count: 1
  1503_markdown_program_descriptions.json

=== ZIP: 1503_markdown_program_descriptions_v2.zip ===
File count: 1
  1503_markdown_program_descriptions_v2.json

=== ZIP: 1503_program_descriptions.zip ===
File count: 1
  1503_program_descriptions.json

=== ZIP: 1503_program_descriptions_v2.zip ===
File count: 1
  1503_program_descriptions_v2.json

=== ZIP: 1503_program_descriptions_x_section.zip ===
File count: 1
  1503_program_descriptions_x_section.csv

=== ZIP: program_descriptions.zip ===
File count: 815
  1503-27672.md
  1503-27674.md
  1503-27675.md
  1503-27676.md
  1503-27677.md
  1503-27678.md
  1503-27679.md
  1503-27680.md
  1503-27681.md
  1503-27682.md
  ...

=== SAMPLE FROM: 1503_markdown_program_descriptions.zip ===
Sample file: 1503_markdown_program_descriptions.json
[
    {
        "page_content": "#  Memorial University of Newfoundland - Anesthesiology - St. John's\n\n#  2025 R-1 Main Re

In [10]:
from pathlib import Path
import zipfile, json, csv, re, hashlib
import pandas as pd

# ---------- paths ----------
def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    while not (p / "pyproject.toml").exists():
        if p.parent == p:
            raise RuntimeError("Could not find repo root")
        p = p.parent
    return p

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

# ---------- helpers ----------
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="replace")).hexdigest()

def normalize_text(s: str) -> str:
    # conservative normalization: unify newlines + trim trailing spaces
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join(line.rstrip() for line in s.split("\n"))
    return s.strip()

def read_zip_single_member_text(zip_path: Path, member_name: str) -> str:
    with zipfile.ZipFile(zip_path) as zf:
        raw = zf.read(member_name)
    return raw.decode("utf-8", errors="replace")

def load_json_list_from_zip(zip_path: Path, member_name: str) -> list[dict]:
    txt = read_zip_single_member_text(zip_path, member_name)
    return json.loads(txt)

def parse_md_files_zip(zip_path: Path) -> dict[str, str]:
    """
    Reads program_descriptions.zip which contains many files like 1503-27672.md
    Returns mapping document_id -> text (document_id = '1503-27672')
    """
    out = {}
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".md"):
                continue
            raw = zf.read(name).decode("utf-8", errors="replace")
            doc_id = Path(name).stem  # '1503-27672'
            out[doc_id] = raw
    return out

def parse_xsection_csv_from_zip(zip_path: Path, member_name: str) -> pd.DataFrame:
    txt = read_zip_single_member_text(zip_path, member_name)
    # csv has an initial empty column header due to index
    from io import StringIO
    df = pd.read_csv(StringIO(txt))
    return df

def v2_id_to_doc_id(v2_id: str) -> str:
    # v2_id like '1503|27447' -> '1503-27447'
    return v2_id.replace("|", "-")

# ---------- locate files ----------
files = {p.name: p for p in DATA_RAW.iterdir() if p.is_file()}

required = [
    "1503_program_master.xlsx",
    "1503_markdown_program_descriptions.zip",
    "1503_markdown_program_descriptions_v2.zip",
    "1503_program_descriptions.zip",
    "1503_program_descriptions_v2.zip",
    "1503_program_descriptions_x_section.zip",
    "program_descriptions.zip",
]
missing = [r for r in required if r not in files]
if missing:
    raise RuntimeError(f"Missing required files in {DATA_RAW}: {missing}")

program_master_path = files["1503_program_master.xlsx"]

md_v1_zip = files["1503_markdown_program_descriptions.zip"]
md_v2_zip = files["1503_markdown_program_descriptions_v2.zip"]
html_v1_zip = files["1503_program_descriptions.zip"]
html_v2_zip = files["1503_program_descriptions_v2.zip"]
xsec_zip = files["1503_program_descriptions_x_section.zip"]
md_files_zip = files["program_descriptions.zip"]

# ---------- load program master ----------
program_df = pd.read_excel(program_master_path)
if "program_stream_id" not in program_df.columns:
    raise RuntimeError(f"Expected program_stream_id in program_master columns, got: {list(program_df.columns)}")

pm_stream_ids = set(program_df["program_stream_id"].dropna().astype(int).astype(str).tolist())
print("Program master program_stream_id count:", len(pm_stream_ids))

# ---------- load JSON v1/v2 (markdown + html) ----------
md_v1 = load_json_list_from_zip(md_v1_zip, "1503_markdown_program_descriptions.json")
md_v2 = load_json_list_from_zip(md_v2_zip, "1503_markdown_program_descriptions_v2.json")
html_v1 = load_json_list_from_zip(html_v1_zip, "1503_program_descriptions.json")
html_v2 = load_json_list_from_zip(html_v2_zip, "1503_program_descriptions_v2.json")

print("\nJSON record counts:")
print(" markdown v1:", len(md_v1))
print(" markdown v2:", len(md_v2))
print(" html v1    :", len(html_v1))
print(" html v2    :", len(html_v2))

# ---------- evidence 1: v2 ids align to program_master ----------
md_v2_ids = [r.get("id") for r in md_v2]
html_v2_ids = [r.get("id") for r in html_v2]

if any(i is None for i in md_v2_ids) or any(i is None for i in html_v2_ids):
    print("\n⚠️ Some v2 records missing 'id' field. First few md_v2 keys:", list(md_v2[0].keys()))
else:
    md_v2_doc_ids = [v2_id_to_doc_id(i) for i in md_v2_ids]
    html_v2_doc_ids = [v2_id_to_doc_id(i) for i in html_v2_ids]

    def extract_stream_id(doc_id: str) -> str:
        # '1503-27447' -> '27447'
        parts = doc_id.split("-")
        return parts[1] if len(parts) == 2 else ""

    md_v2_streams = {extract_stream_id(d) for d in md_v2_doc_ids}
    html_v2_streams = {extract_stream_id(d) for d in html_v2_doc_ids}

    md_missing = sorted(list(md_v2_streams - pm_stream_ids))[:20]
    html_missing = sorted(list(html_v2_streams - pm_stream_ids))[:20]

    print("\n=== Evidence 1: v2 IDs align to program_master ===")
    print("markdown v2 unique stream ids:", len(md_v2_streams))
    print("html v2 unique stream ids    :", len(html_v2_streams))
    print("markdown v2 stream ids missing from program_master:", len(md_v2_streams - pm_stream_ids))
    if md_missing:
        print("  sample missing:", md_missing)
    print("html v2 stream ids missing from program_master    :", len(html_v2_streams - pm_stream_ids))
    if html_missing:
        print("  sample missing:", html_missing)

# ---------- evidence 2: markdown v2 json matches md files zip ----------
md_files = parse_md_files_zip(md_files_zip)  # doc_id -> text
print("\nMD files count in program_descriptions.zip:", len(md_files))

# Build hash maps
md_v2_by_doc = {}
for r in md_v2:
    vid = r.get("id")
    pc = r.get("page_content", "")
    if not vid:
        continue
    doc_id = v2_id_to_doc_id(vid)
    md_v2_by_doc[doc_id] = sha256_text(normalize_text(pc))

md_files_hash = {doc_id: sha256_text(normalize_text(txt)) for doc_id, txt in md_files.items()}

common_doc_ids = sorted(set(md_v2_by_doc.keys()) & set(md_files_hash.keys()))
hash_matches = sum(1 for d in common_doc_ids if md_v2_by_doc[d] == md_files_hash[d])
hash_mismatches = [d for d in common_doc_ids if md_v2_by_doc[d] != md_files_hash[d]]

print("\n=== Evidence 2: markdown v2 JSON == md files (hash) ===")
print("Common doc_ids:", len(common_doc_ids))
print("Hash matches  :", hash_matches)
print("Hash mismatches:", len(hash_mismatches))
if hash_mismatches:
    print("  sample mismatches:", hash_mismatches[:10])

# ---------- evidence 3: v1 vs v2 content overlap (markdown + html) ----------
# v1 lacks id, so we compare as multisets of content hashes.
md_v1_hashes = [sha256_text(normalize_text(r.get("page_content",""))) for r in md_v1]
md_v2_hashes = [sha256_text(normalize_text(r.get("page_content",""))) for r in md_v2]
html_v1_hashes = [sha256_text(normalize_text(r.get("page_content",""))) for r in html_v1]
html_v2_hashes = [sha256_text(normalize_text(r.get("page_content",""))) for r in html_v2]

md_v1_set = set(md_v1_hashes); md_v2_set = set(md_v2_hashes)
html_v1_set = set(html_v1_hashes); html_v2_set = set(html_v2_hashes)

print("\n=== Evidence 3: v1 vs v2 are same content (set overlap) ===")
print("Markdown v1 unique hashes:", len(md_v1_set))
print("Markdown v2 unique hashes:", len(md_v2_set))
print("Markdown overlap         :", len(md_v1_set & md_v2_set), "/", len(md_v1_set | md_v2_set))

print("HTML v1 unique hashes    :", len(html_v1_set))
print("HTML v2 unique hashes    :", len(html_v2_set))
print("HTML overlap             :", len(html_v1_set & html_v2_set), "/", len(html_v1_set | html_v2_set))

# ---------- evidence 4: x_section document_id aligns ----------
xsec_df = parse_xsection_csv_from_zip(xsec_zip, "1503_program_descriptions_x_section.csv")
# document_id column exists per your sample
if "document_id" not in xsec_df.columns:
    print("\n⚠️ x_section csv missing 'document_id' column. Columns:", list(xsec_df.columns))
else:
    xsec_doc_ids = set(xsec_df["document_id"].dropna().astype(str).tolist())
    # Compare to markdown v2 doc ids
    md_v2_doc_ids = set(md_v2_by_doc.keys()) if md_v2_by_doc else set()
    overlap = xsec_doc_ids & md_v2_doc_ids

    print("\n=== Evidence 4: x_section aligns to same doc_id space ===")
    print("x_section rows:", len(xsec_df))
    print("x_section unique doc_ids:", len(xsec_doc_ids))
    print("overlap with markdown v2 doc_ids:", len(overlap))
    if overlap:
        # show a couple example rows for overlapped ids
        sample_ids = list(sorted(overlap))[:3]
        print("sample overlapped doc_ids:", sample_ids)
        display(xsec_df[xsec_df["document_id"].isin(sample_ids)].head(3))


Program master program_stream_id count: 815

JSON record counts:
 markdown v1: 815
 markdown v2: 815
 html v1    : 815
 html v2    : 815

=== Evidence 1: v2 IDs align to program_master ===
markdown v2 unique stream ids: 815
html v2 unique stream ids    : 815
markdown v2 stream ids missing from program_master: 0
html v2 stream ids missing from program_master    : 0

MD files count in program_descriptions.zip: 815

=== Evidence 2: markdown v2 JSON == md files (hash) ===
Common doc_ids: 815
Hash matches  : 0
Hash mismatches: 815
  sample mismatches: ['1503-26260', '1503-26262', '1503-26263', '1503-26265', '1503-26266', '1503-26267', '1503-26268', '1503-26271', '1503-26272', '1503-26276']

=== Evidence 3: v1 vs v2 are same content (set overlap) ===
Markdown v1 unique hashes: 815
Markdown v2 unique hashes: 815
Markdown overlap         : 811 / 819
HTML v1 unique hashes    : 815
HTML v2 unique hashes    : 815
HTML overlap             : 811 / 819

=== Evidence 4: x_section aligns to same doc_i

,Unnamed: 0,document_id,source,n_program_description_sections,program_name,match_iteration_name,program_contracts,general_instructions,supporting_documentation_information,review_process,...,selection_criteria,program_highlights,program_curriculum,training_sites,additional_information,return_of_service,faq,summary_of_changes,match_iteration_id,program_description_id
217,217,1503-26260,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...,15,# University of Ottawa - Family Medicine - Ru...,# 2025 R-1 Main Residency Match - first itera...,# Program Contacts \nName: Kim Rozon \nTitle...,# General Instructions \nProgram application ...,# Supporting Documentation / Information \nCa...,# Review Process \nApplications submitted aft...,...,# Selection Criteria \n**Pre-Residency Progra...,# Program Highlights \nAre you a new graduate...,# Program Curriculum \nThis residency program...,# Training Sites \nOur rural stream is locate...,# Additional Information \nApplications may b...,# Return of Service \nOntario's International...,NaN,# Summary of changes \nSUMMARY ID | Section |...,1503,26260
465,465,1503-26263,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...,16,# University of Alberta - Internal Medicine -...,# 2025 R-1 Main Residency Match - first itera...,# Program Contacts \nName: Jennifer Aquin \n...,# General Instructions \nProgram application ...,# Supporting Documentation / Information \nAp...,# Review Process \nApplications submitted aft...,...,# Selection Criteria \nSelection for Intervie...,# Program Highlights \n**Highlights of the In...,# Program Curriculum \nThis residency program...,# Training Sites \n* University of Alberta H...,# Additional Information \nCaRMS Self-Identif...,# Return of Service \nThere are no return of ...,('# FAQ \n**Can I meet with the Program Direc...,# Summary of changes \nSUMMARY ID | Section |...,1503,26263
648,648,1503-26262,https://phx.e-carms.ca/phoenix-web/pd/ajax/pro...,13,# University of Toronto - Pediatrics - Toronto,# 2025 R-1 Main Residency Match - first itera...,# Program Contacts \nName: Dr. Kevin Weingart...,# General Instructions \nProgram application ...,# Supporting Documentation / Information \nCa...,# Review Process \nApplications submitted aft...,...,# Selection Criteria \n**Philosophy ** \nTh...,# Program Highlights \nThe Paediatric Residen...,# Program Curriculum \nThis residency program...,# Training Sites \n* Holland Bloorview Kids R...,# Additional Information \n1. **Permanent Res...,NaN,NaN,NaN,1503,26262


In [11]:
import difflib

# We already have md_v1, md_v2, html_v1, html_v2 lists of dicts from earlier cell
# And normalize_text, sha256_text, v2_id_to_doc_id helpers exist

def index_by_hash(records):
    """hash -> list of (idx, normalized_text)"""
    out = {}
    for i, r in enumerate(records):
        txt = normalize_text(r.get("page_content",""))
        h = sha256_text(txt)
        out.setdefault(h, []).append((i, txt))
    return out

def index_v2_by_hash_and_id(records_v2):
    """hash -> list of (doc_id, normalized_text)"""
    out = {}
    for r in records_v2:
        vid = r.get("id")
        if not vid:
            continue
        doc_id = v2_id_to_doc_id(vid)
        txt = normalize_text(r.get("page_content",""))
        h = sha256_text(txt)
        out.setdefault(h, []).append((doc_id, txt))
    return out

def show_diff(a, b, title_a="v1", title_b="v2", max_lines=120):
    a_lines = a.splitlines()
    b_lines = b.splitlines()
    diff = list(difflib.unified_diff(a_lines, b_lines, fromfile=title_a, tofile=title_b, lineterm=""))
    print("\n".join(diff[:max_lines]))
    if len(diff) > max_lines:
        print(f"... (diff truncated, total lines {len(diff)})")

def analyze_version_deltas(v1_records, v2_records, label):
    v1_idx = index_by_hash(v1_records)
    v2_idx = index_by_hash(v2_records)

    v1_set = set(v1_idx.keys())
    v2_set = set(v2_idx.keys())

    only_v1 = sorted(v1_set - v2_set)
    only_v2 = sorted(v2_set - v1_set)

    print(f"\n=== {label}: hashes only in v1: {len(only_v1)}, only in v2: {len(only_v2)} ===")
    return v1_idx, v2_idx, only_v1, only_v2

# Analyze markdown and HTML separately
md_v1_idx, md_v2_idx, md_only_v1, md_only_v2 = analyze_version_deltas(md_v1, md_v2, "MARKDOWN")
html_v1_idx, html_v2_idx, html_only_v1, html_only_v2 = analyze_version_deltas(html_v1, html_v2, "HTML")

# For v2, we can map hash -> doc_id(s) using v2 IDs (better evidence)
md_v2_hash_to_doc = index_v2_by_hash_and_id(md_v2)
html_v2_hash_to_doc = index_v2_by_hash_and_id(html_v2)

def inspect_only_hashes(v1_idx, v2_idx, only_v1, only_v2, v2_hash_to_doc, label):
    # show up to 4 examples (since you saw 4)
    n = min(4, len(only_v1), len(only_v2))
    print(f"\n--- {label}: showing {n} v1-only and {n} v2-only examples ---")

    for k in range(n):
        h1 = only_v1[k]
        h2 = only_v2[k]

        v1_txt = v1_idx[h1][0][1]  # first occurrence
        v2_txt = v2_idx[h2][0][1]

        doc_info = v2_hash_to_doc.get(h2, [])
        doc_str = ", ".join(d for d, _ in doc_info[:3]) if doc_info else "(unknown doc_id)"
        print(f"\n[{label}] v2-only hash example {k+1} doc_id(s): {doc_str}")

        # Print small heads so you can visually see if it’s a header/footer thing
        print("\n[v1-only text head]")
        print(v1_txt[:400])
        print("\n[v2-only text head]")
        print(v2_txt[:400])

        # Diff
        print("\n[diff]")
        show_diff(v1_txt, v2_txt, title_a="v1_only_sample", title_b="v2_only_sample", max_lines=120)

inspect_only_hashes(md_v1_idx, md_v2_idx, md_only_v1, md_only_v2, md_v2_hash_to_doc, "MARKDOWN")
inspect_only_hashes(html_v1_idx, html_v2_idx, html_only_v1, html_only_v2, html_v2_hash_to_doc, "HTML")



=== MARKDOWN: hashes only in v1: 4, only in v2: 4 ===

=== HTML: hashes only in v1: 4, only in v2: 4 ===

--- MARKDOWN: showing 4 v1-only and 4 v2-only examples ---

[MARKDOWN] v2-only hash example 1 doc_id(s): 1503-26429

[v1-only text head]
#  University of Ottawa - Diagnostic Radiology - Ottawa

#  2025 R-1 Main Residency Match - first iteration
IMG Stream for IMG

##  Last approved on October 09, 2024

Summary of changes

## Approximate Quota:

##   1 click here

##  Accreditation status : Accredited

##  Provincial Criteria

Print copy link

Link to display this program description

To copy link, press Ctrl-C on your keyboard or r

[v2-only text head]
#  University of Ottawa - Diagnostic Radiology - Ottawa

#  2025 R-1 Main Residency Match - first iteration
IMG Stream for IMG

##  Last approved on October 09, 2024

Summary of changes

## Approximate Quota:

##   1 click here

##  Accreditation status : Accredited

##  Provincial Criteria

Print copy link

Link to display this pro

In [12]:
## Identify the exact doc_ids that differ
import json, zipfile, hashlib
from pathlib import Path
import pandas as pd
import difflib

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    while not (p / "pyproject.toml").exists():
        if p.parent == p:
            raise RuntimeError("Could not find repo root")
        p = p.parent
    return p

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

def normalize_text(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join(line.rstrip() for line in s.split("\n"))
    return s.strip()

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="replace")).hexdigest()

def v2_id_to_doc_id(v2_id: str) -> str:
    return v2_id.replace("|", "-")

def load_json_from_zip(zip_path: Path, member: str):
    with zipfile.ZipFile(zip_path) as zf:
        txt = zf.read(member).decode("utf-8", errors="replace")
    return json.loads(txt)

def load_md_files_from_zip(zip_path: Path) -> dict[str, str]:
    out = {}
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.lower().endswith(".md"):
                doc_id = Path(name).stem  # 1503-27672
                out[doc_id] = zf.read(name).decode("utf-8", errors="replace")
    return out

def unified_diff(a: str, b: str, title_a: str, title_b: str, max_lines=120):
    a_lines = a.splitlines()
    b_lines = b.splitlines()
    diff = list(difflib.unified_diff(a_lines, b_lines, fromfile=title_a, tofile=title_b, lineterm=""))
    print("\n".join(diff[:max_lines]))
    if len(diff) > max_lines:
        print(f"... (diff truncated, total lines {len(diff)})")

# Load v2 markdown JSON (keyed)
md_v2_zip = DATA_RAW / "1503_markdown_program_descriptions_v2.zip"
md_v2 = load_json_from_zip(md_v2_zip, "1503_markdown_program_descriptions_v2.json")

md_v2_by_doc = {}
for r in md_v2:
    doc_id = v2_id_to_doc_id(r["id"])
    md_v2_by_doc[doc_id] = normalize_text(r.get("page_content", ""))

# Load md files zip (keyed by filename)
md_files_zip = DATA_RAW / "program_descriptions.zip"
md_files_by_doc = {k: normalize_text(v) for k, v in load_md_files_from_zip(md_files_zip).items()}

common = sorted(set(md_v2_by_doc) & set(md_files_by_doc))
mismatched = [d for d in common if sha256_text(md_v2_by_doc[d]) != sha256_text(md_files_by_doc[d])]

print("=== Canonical check: md_v2 JSON vs program_descriptions.zip ===")
print("Common doc_ids:", len(common))
print("Mismatches:", len(mismatched))
if mismatched:
    print("Mismatched doc_ids:", mismatched)

# OPTIONAL: map v1 markdown JSON to doc_id by content hash matching
md_v1_zip = DATA_RAW / "1503_markdown_program_descriptions.zip"
md_v1 = load_json_from_zip(md_v1_zip, "1503_markdown_program_descriptions.json")

# Build hash -> doc_id from md files (canonical)
md_files_hash_to_doc = {}
for doc_id, txt in md_files_by_doc.items():
    md_files_hash_to_doc.setdefault(sha256_text(txt), []).append(doc_id)

# Map v1 records onto doc_ids if possible
v1_mapped = []
unmapped = 0
ambiguous = 0

for r in md_v1:
    txt = normalize_text(r.get("page_content", ""))
    h = sha256_text(txt)
    candidates = md_files_hash_to_doc.get(h, [])
    if len(candidates) == 1:
        v1_mapped.append((candidates[0], txt))
    elif len(candidates) == 0:
        unmapped += 1
    else:
        ambiguous += 1  # same text appears in multiple doc_ids (rare but possible)

print("\n=== Mapping v1 markdown to doc_ids by content hash ===")
print("v1 records:", len(md_v1))
print("mapped uniquely:", len(v1_mapped))
print("unmapped:", unmapped)
print("ambiguous:", ambiguous)

# Compare v1-mapped vs v2 per doc_id to find true content changes (not pairing artifacts)
v1_by_doc = dict(v1_mapped)
common_v1_v2 = sorted(set(v1_by_doc) & set(md_v2_by_doc))
true_deltas = [d for d in common_v1_v2 if sha256_text(v1_by_doc[d]) != sha256_text(md_v2_by_doc[d])]

print("\n=== True per-doc content deltas: v1 vs v2 (markdown) ===")
print("Docs comparable:", len(common_v1_v2))
print("Docs changed:", len(true_deltas))
if true_deltas:
    print("Changed doc_ids:", true_deltas[:20])

    # show one example diff
    d0 = true_deltas[0]
    print(f"\n--- Diff for {d0} ---")
    unified_diff(v1_by_doc[d0], md_v2_by_doc[d0], "v1", "v2", max_lines=120)


=== Canonical check: md_v2 JSON vs program_descriptions.zip ===
Common doc_ids: 815
Mismatches: 815
Mismatched doc_ids: ['1503-26260', '1503-26262', '1503-26263', '1503-26265', '1503-26266', '1503-26267', '1503-26268', '1503-26271', '1503-26272', '1503-26276', '1503-26278', '1503-26280', '1503-26281', '1503-26284', '1503-26285', '1503-26286', '1503-26288', '1503-26291', '1503-26292', '1503-26293', '1503-26294', '1503-26297', '1503-26298', '1503-26300', '1503-26302', '1503-26303', '1503-26306', '1503-26307', '1503-26309', '1503-26310', '1503-26312', '1503-26313', '1503-26314', '1503-26315', '1503-26317', '1503-26318', '1503-26324', '1503-26325', '1503-26327', '1503-26329', '1503-26330', '1503-26331', '1503-26333', '1503-26334', '1503-26336', '1503-26337', '1503-26338', '1503-26339', '1503-26343', '1503-26344', '1503-26345', '1503-26346', '1503-26347', '1503-26348', '1503-26349', '1503-26350', '1503-26352', '1503-26353', '1503-26354', '1503-26355', '1503-26357', '1503-26358', '1503-26360

In [14]:
import zipfile, json, difflib, re
from pathlib import Path

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

md_v2_zip = DATA_RAW / "1503_markdown_program_descriptions_v2.zip"
md_files_zip = DATA_RAW / "program_descriptions.zip"

def load_json_from_zip(zip_path: Path, member: str):
    with zipfile.ZipFile(zip_path) as zf:
        txt = zf.read(member).decode("utf-8", errors="replace")
    return json.loads(txt)

def read_md_from_zip(zip_path: Path, doc_id: str) -> str:
    member = f"{doc_id}.md"
    with zipfile.ZipFile(zip_path) as zf:
        raw = zf.read(member)
    return raw.decode("utf-8", errors="replace")

def norm_basic(s: str) -> str:
    return s.replace("\r\n", "\n").replace("\r", "\n")

def norm_aggressive_ws(s: str) -> str:
    # collapse all whitespace to single spaces for "same content" check
    s = norm_basic(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

md_v2 = load_json_from_zip(md_v2_zip, "1503_markdown_program_descriptions_v2.json")

# choose a doc_id from v2 (first record)
doc_id = md_v2[0]["id"].replace("|", "-")
json_txt = md_v2[0].get("page_content", "")
md_txt = read_md_from_zip(md_files_zip, doc_id)

json_txt_b = norm_basic(json_txt)
md_txt_b = norm_basic(md_txt)

print("Doc:", doc_id)
print("JSON length:", len(json_txt_b))
print("MD length  :", len(md_txt_b))
print("\nJSON head:\n", json_txt_b[:300])
print("\nMD head:\n", md_txt_b[:300])

# Compare aggressive whitespace to see if it's mainly formatting
same_ws_insensitive = norm_aggressive_ws(json_txt) == norm_aggressive_ws(md_txt)
print("\nWhitespace-insensitive equal?:", same_ws_insensitive)

# Show a diff sample
diff = list(difflib.unified_diff(
    json_txt_b.splitlines(),
    md_txt_b.splitlines(),
    fromfile="json_page_content",
    tofile="md_file",
    lineterm=""
))
print("\n--- DIFF (first 200 lines) ---")
print("\n".join(diff[:200]))
if len(diff) > 200:
    print(f"... (diff truncated, total diff lines {len(diff)})")


Doc: 1503-27447
JSON length: 25056
MD length  : 26605

JSON head:
 #  Memorial University of Newfoundland - Anesthesiology - St. John's

#  2025 R-1 Main Residency Match - first iteration  
CMG Stream for CMG  

##  Last approved on February 04, 2025

Summary of changes

## Approximate Quota:

##   4

##  Accreditation status : Accredited

##  Provincial Criteria



MD head:
 #  Memorial University of Newfoundland - Anesthesiology - St. John's

#  2025 R-1 Main Residency Match - first iteration  
CMG Stream for CMG  

##  Last approved on February 04, 2025

Summary of changes

## Approximate Quota: 4

##  Accreditation status : Accredited

##  [Provincial Criteria](https

Whitespace-insensitive equal?: False

--- DIFF (first 200 lines) ---
--- json_page_content
+++ md_file
@@ -7,13 +7,12 @@
 
 Summary of changes
 
-## Approximate Quota:
-
-##   4
+## Approximate Quota: 4
 
 ##  Accreditation status : Accredited
 
-##  Provincial Criteria
+##  [Provincial Criteria](https://www.carms.ca/

In [15]:
import random
import difflib
import zipfile, json
from pathlib import Path

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

md_v2_zip = DATA_RAW / "1503_markdown_program_descriptions_v2.zip"
md_files_zip = DATA_RAW / "program_descriptions.zip"

def load_json_from_zip(zip_path: Path, member: str):
    with zipfile.ZipFile(zip_path) as zf:
        txt = zf.read(member).decode("utf-8", errors="replace")
    return json.loads(txt)

def read_md_from_zip(zip_path: Path, doc_id: str) -> str:
    member = f"{doc_id}.md"
    with zipfile.ZipFile(zip_path) as zf:
        raw = zf.read(member)
    return raw.decode("utf-8", errors="replace")

def v2_id_to_doc_id(v2_id: str) -> str:
    return v2_id.replace("|", "-")

def norm_basic(s: str) -> str:
    return s.replace("\r\n", "\n").replace("\r", "\n").strip()

md_v2 = load_json_from_zip(md_v2_zip, "1503_markdown_program_descriptions_v2.json")
doc_ids = [v2_id_to_doc_id(r["id"]) for r in md_v2]

# sample size (tune up/down)
N = 30
sample = random.sample(doc_ids, k=min(N, len(doc_ids)))

scores = []
worst = None

for doc_id in sample:
    # get corresponding json record
    r = next(x for x in md_v2 if v2_id_to_doc_id(x["id"]) == doc_id)
    a = norm_basic(r.get("page_content",""))
    b = norm_basic(read_md_from_zip(md_files_zip, doc_id))

    ratio = difflib.SequenceMatcher(None, a, b).ratio()
    scores.append((doc_id, ratio, len(a), len(b)))

    if (worst is None) or (ratio < worst[1]):
        worst = (doc_id, ratio, a, b)

avg = sum(r for _, r, _, _ in scores) / len(scores)
mn = min(r for _, r, _, _ in scores)
mx = max(r for _, r, _, _ in scores)

print(f"Sample size: {len(scores)}")
print(f"Similarity ratio: avg={avg:.3f} min={mn:.3f} max={mx:.3f}")

# show top 5 lowest similarity docs in sample
scores_sorted = sorted(scores, key=lambda x: x[1])
print("\nLowest 5 in sample:")
for doc_id, ratio, la, lb in scores_sorted[:5]:
    print(f"  {doc_id}: ratio={ratio:.3f} (len json={la}, len md={lb})")

# print a short diff for the worst doc in sample (first 120 diff lines)
doc_id, ratio, a, b = worst
print(f"\nWorst doc in sample: {doc_id}, ratio={ratio:.3f}")
diff = list(difflib.unified_diff(a.splitlines(), b.splitlines(), fromfile="json_md_v2", tofile="md_file", lineterm=""))
print("\n".join(diff[:120]))
if len(diff) > 120:
    print(f"... (diff truncated, total diff lines {len(diff)})")


Sample size: 30
Similarity ratio: avg=0.929 min=0.741 max=0.971

Lowest 5 in sample:
  1503-27026: ratio=0.741 (len json=42672, len md=44780)
  1503-27274: ratio=0.860 (len json=27365, len md=29557)
  1503-26906: ratio=0.864 (len json=21575, len md=23528)
  1503-27351: ratio=0.888 (len json=36753, len md=39749)
  1503-26288: ratio=0.899 (len json=20472, len md=22682)

Worst doc in sample: 1503-27026, ratio=0.741
--- json_md_v2
+++ md_file
@@ -7,13 +7,12 @@
 
 Summary of changes
 
-## Approximate Quota:
-
-##   4
+## Approximate Quota: 4
 
 ##  Accreditation status : Accredited
 
-##  Provincial Criteria
+##  [Provincial Criteria](https://www.carms.ca/match/r-1-main-residency-
+match/eligibility-criteria/)
 
 Print copy link
 
@@ -31,8 +30,10 @@
 P228-770 Bannatyne Avenue  
 Winnipeg, Manitoba, R3E 0W3  
 
-Websites of Interest About Our Program:  
-Provincial Criteria for Manitoba:  
+Websites of Interest [ About Our Program:
+](http://umanitoba.ca/healthsciences/medicine/units/family_

Test Cell for Proposed Database Schema

In [1]:
from __future__ import annotations

import os, json, zipfile, hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from sqlmodel import SQLModel, Field, create_engine, Session
from sqlalchemy import Column, DateTime, UniqueConstraint, text
from sqlalchemy.dialects.postgresql import insert as pg_insert

# ------------------------------------------------------------
# Paths + env
# ------------------------------------------------------------
def find_repo_root(start=None) -> Path:
    p = Path(start or Path.cwd()).resolve()
    while not (p / "pyproject.toml").exists():
        if p.parent == p:
            raise RuntimeError("Could not find repo root")
        p = p.parent
    return p

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw" / "dnokes"

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL not set. Ensure .env exists.")

engine = create_engine(DATABASE_URL, echo=False)

MATCH_CYCLE_ID = 1503

# ------------------------------------------------------------
# Schema (same as before, embedded here so notebook is standalone)
# ------------------------------------------------------------
class Discipline(SQLModel, table=True):
    __tablename__ = "discipline"
    discipline_id: int = Field(primary_key=True)
    discipline: str

class ProgramStream(SQLModel, table=True):
    __tablename__ = "program_stream"
    program_stream_id: int = Field(primary_key=True)
    match_cycle_id: int = Field(index=True)

    discipline_id: int | None = Field(default=None, foreign_key="discipline.discipline_id", index=True)
    discipline_name: str | None = Field(default=None)

    school_id: int | None = Field(default=None, index=True)
    school_name: str | None = Field(default=None)

    program_stream_id_src: int | None = Field(default=None)  # optional extra provenance
    program_stream_name: str | None = Field(default=None)
    program_site: str | None = Field(default=None)
    program_stream: str | None = Field(default=None)

    program_name: str | None = Field(default=None)
    program_url: str | None = Field(default=None)

    row_index: int | None = Field(default=None)

class ProgramDescriptionArtifact(SQLModel, table=True):
    __tablename__ = "program_description_artifact"
    __table_args__ = (UniqueConstraint("doc_id", "representation", name="uq_doc_rep"),)

    artifact_id: int | None = Field(default=None, primary_key=True)

    doc_id: str = Field(index=True)                 # "1503-27447"
    match_cycle_id: int = Field(index=True)         # 1503
    program_stream_id: int = Field(foreign_key="program_stream.program_stream_id", index=True)

    representation: str = Field(index=True)         # "markdown_v2"
    source_file: str | None = Field(default=None)

    content: str
    content_sha256: str | None = Field(default=None, index=True)

    ingested_at: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc),
        sa_column=Column(DateTime(timezone=True), nullable=False),
    )

# Create tables if missing
SQLModel.metadata.create_all(engine)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="replace")).hexdigest()

def read_json_from_zip(zip_path: Path, member: str):
    with zipfile.ZipFile(zip_path) as zf:
        raw = zf.read(member).decode("utf-8", errors="replace")
    return json.loads(raw)

def upsert_dataframe(session: Session, table, rows: list[dict], conflict_cols: list[str], update_cols: list[str]):
    if not rows:
        return 0
    stmt = pg_insert(table).values(rows)
    stmt = stmt.on_conflict_do_update(
        index_elements=conflict_cols,
        set_={c: getattr(stmt.excluded, c) for c in update_cols}
    )
    res = session.exec(stmt)
    # SQLModel Session.exec returns result proxy-like; rowcount may be -1 depending on driver
    return len(rows)

# ------------------------------------------------------------
# Load discipline
# ------------------------------------------------------------
discipline_df = pd.read_excel(DATA_RAW / "1503_discipline.xlsx")
# Expect columns discipline_id, discipline
disc_rows = []
for _, r in discipline_df.iterrows():
    disc_rows.append({
        "discipline_id": int(r["discipline_id"]),
        "discipline": str(r["discipline"]),
    })

# ------------------------------------------------------------
# Load program master
# ------------------------------------------------------------
program_df = pd.read_excel(DATA_RAW / "1503_program_master.xlsx")

# Normalize Unnamed column
row_index_col = None
for c in program_df.columns:
    if str(c).strip().lower().startswith("unnamed"):
        row_index_col = c
        break

prog_rows = []
for _, r in program_df.iterrows():
    prog_rows.append({
        "program_stream_id": int(r["program_stream_id"]),
        "match_cycle_id": MATCH_CYCLE_ID,
        "discipline_id": int(r["discipline_id"]) if pd.notna(r["discipline_id"]) else None,
        "discipline_name": str(r["discipline_name"]) if pd.notna(r["discipline_name"]) else None,
        "school_id": int(r["school_id"]) if pd.notna(r["school_id"]) else None,
        "school_name": str(r["school_name"]) if pd.notna(r["school_name"]) else None,
        "program_stream_name": str(r["program_stream_name"]) if pd.notna(r["program_stream_name"]) else None,
        "program_site": str(r["program_site"]) if pd.notna(r["program_site"]) else None,
        "program_stream": str(r["program_stream"]) if pd.notna(r["program_stream"]) else None,
        "program_name": str(r["program_name"]) if pd.notna(r["program_name"]) else None,
        "program_url": str(r["program_url"]) if pd.notna(r["program_url"]) else None,
        "row_index": int(r[row_index_col]) if row_index_col and pd.notna(r[row_index_col]) else None,
    })

# ------------------------------------------------------------
# Load markdown v2 descriptions (keyed)
# ------------------------------------------------------------
md_v2_zip = DATA_RAW / "1503_markdown_program_descriptions_v2.zip"
md_v2 = read_json_from_zip(md_v2_zip, "1503_markdown_program_descriptions_v2.json")

artifact_rows = []
missing_stream = 0

for rec in md_v2:
    vid = rec.get("id")  # "1503|27447"
    content = rec.get("page_content", "")
    if not vid:
        continue
    doc_id = vid.replace("|", "-")  # "1503-27447"
    parts = doc_id.split("-")
    if len(parts) != 2:
        continue
    stream_id = int(parts[1])

    # sanity: only insert if stream exists in program_master universe
    # (we still allow insertion, but we count missing)
    pm_ids = set(program_df["program_stream_id"].astype(int).tolist())
    if stream_id not in pm_ids:
        missing_stream += 1

    artifact_rows.append({
        "doc_id": doc_id,
        "match_cycle_id": MATCH_CYCLE_ID,
        "program_stream_id": stream_id,
        "representation": "markdown_v2",
        "source_file": md_v2_zip.name,
        "content": content,
        "content_sha256": sha256_text(content),
        "ingested_at": datetime.now(timezone.utc),
    })

# ------------------------------------------------------------
# Upsert into DB
# ------------------------------------------------------------
with Session(engine) as session:
    # discipline upsert (conflict on PK)
    upsert_dataframe(
        session,
        Discipline.__table__,
        disc_rows,
        conflict_cols=["discipline_id"],
        update_cols=["discipline"],
    )

    # program_stream upsert (conflict on PK)
    upsert_dataframe(
        session,
        ProgramStream.__table__,
        prog_rows,
        conflict_cols=["program_stream_id"],
        update_cols=[
            "match_cycle_id","discipline_id","discipline_name","school_id","school_name",
            "program_stream_name","program_site","program_stream","program_name","program_url","row_index"
        ],
    )

    # artifact upsert (conflict on uq_doc_rep)
    upsert_dataframe(
        session,
        ProgramDescriptionArtifact.__table__,
        artifact_rows,
        conflict_cols=["doc_id","representation"],
        update_cols=[
            "match_cycle_id","program_stream_id","source_file","content","content_sha256","ingested_at"
        ],
    )

    session.commit()

print("Loaded:")
print(f"  discipline rows: {len(disc_rows)}")
print(f"  program_stream rows: {len(prog_rows)}")
print(f"  markdown_v2 artifacts: {len(artifact_rows)}")
print(f"  artifacts w/ stream_id not in program_master: {missing_stream}")

# ------------------------------------------------------------
# Test pull: simple SQL sanity checks
# ------------------------------------------------------------
with engine.connect() as conn:
    n_disc = conn.execute(text("select count(*) from discipline")).scalar()
    n_prog = conn.execute(text("select count(*) from program_stream")).scalar()
    n_art  = conn.execute(text("select count(*) from program_description_artifact")).scalar()

    print("\nDB counts:")
    print("  discipline:", n_disc)
    print("  program_stream:", n_prog)
    print("  program_description_artifact:", n_art)

    print("\nSample join (first 5):")
    rows = conn.execute(text("""
        select p.program_stream_id, p.school_name, p.discipline_name, a.representation,
               left(a.content, 120) as content_head
        from program_stream p
        join program_description_artifact a on a.program_stream_id = p.program_stream_id
        where a.representation = 'markdown_v2'
        order by p.program_stream_id
        limit 5;
    """)).fetchall()

for r in rows:
    print(r)


Loaded:
  discipline rows: 37
  program_stream rows: 815
  markdown_v2 artifacts: 815
  artifacts w/ stream_id not in program_master: 0

DB counts:
  discipline: 37
  program_stream: 815
  program_description_artifact: 815

Sample join (first 5):
(26260, 'University of Ottawa', 'Family Medicine', 'markdown_v2', '#  University of Ottawa - Family Medicine - Rural Pembroke\n\n#  2025 R-1 Main Residency Match - first iteration  \nIMG Str')
(26262, 'University of Toronto', 'Pediatrics', 'markdown_v2', '#  University of Toronto - Pediatrics - Toronto\n\n#  2025 R-1 Main Residency Match - first iteration  \nCMG Stream for CMG')
(26263, 'University of Alberta', 'Internal Medicine', 'markdown_v2', '#  University of Alberta - Internal Medicine - Edmonton  \n  \n#  2025 R-1 Main Residency Match - first iteration  \nCMG St')
(26265, 'McMaster University', 'Internal Medicine', 'markdown_v2', '#  McMaster University - Internal Medicine - Waterloo  \n  \n#  2025 R-1 Main Residency Match - first itera

In [2]:
from sqlalchemy import text
from carms_platform.db import engine

with engine.begin() as conn:
    conn.execute(text("""
    create or replace view v_program_description_canonical as
    select
        a.program_stream_id,
        a.doc_id,
        a.match_cycle_id,
        a.representation,
        a.content
    from program_description_artifact a
    where a.representation = 'markdown_v2';
    """))
print("Created view: v_program_description_canonical")


Created view: v_program_description_canonical


In [3]:
from sqlalchemy import text
from carms_platform.db import engine

with engine.begin() as conn:
    rows = conn.execute(text("""
        select program_stream_id, school_name, discipline_name
        from program_stream
        order by program_stream_id
        limit 10;
    """)).fetchall()

for r in rows:
    print(r)


(26260, 'University of Ottawa', 'Family Medicine')
(26262, 'University of Toronto', 'Pediatrics')
(26263, 'University of Alberta', 'Internal Medicine')
(26265, 'McMaster University', 'Internal Medicine')
(26266, 'Université Laval', 'Vascular Surgery')
(26267, 'McGill University', 'Internal Medicine')
(26268, 'University of Manitoba', 'Family Medicine')
(26271, 'University of British Columbia', 'Family Medicine')
(26272, 'University of Alberta', 'Medical Microbiology')
(26276, 'McGill University', 'Neurosurgery')
